# Assignment 1: Safety Stock, Data Cleaning and Transformation

**Group:** 
Daniel Ramirez,
Diego Holguin,
Jorge Hernandez Ramirez,
Suleiman Sahawneh.

Throughout our assignments we calculate the safety stock value for individual SKUs designated as make-to-stock. This assignment deals with the initial task: cleaning the transaction data and performing some useful transformations.

# Setup

We use `pandas` for loading and cleaning the data and `numpy` for `np.where` in Question 4.

In [1]:
import pandas as pd
import numpy as np

## Loading the data

Our group's dataset (`transaction_data.csv`) and its metadata (`meta_data.csv`) are in the `data` subfolder.

In [2]:
transactions = pd.read_csv("../data/transaction_data.csv")
metadata = pd.read_csv("../data/meta_data.csv")
transactions.head()

,sku_numb3r,inventory_type,stocking_type,lead_time,unit_price,Manufacturing Site,division_code,transaction_date,Order-Quantity
0,C4CCB7BB,FG,MTS,28,1014.553490,US-1F,AF8D,2023-01-01,50
1,1CC9F4D5,FG,MTS,28,1014.698985,NaN,A873,2023-01-01,105
2,88D82D16,WIP,MTS,28,1003.032674,PL-FA,8CF4,2023-01-01,105
3,5DDF2B93,FG,MTO,28,1009.165291,CA-D0,1ED9,2023-01-01,129
4,20A7E2D1,RM,MTS,28,1007.309891,US-F1,ECB9,2023-01-01,133


In [3]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 728463 entries, 0 to 728462
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   sku_numb3r          724354 non-null  str    
 1   inventory_type      713832 non-null  str    
 2   stocking_type       721540 non-null  str    
 3   lead_time           728463 non-null  int64  
 4   unit_price          728463 non-null  float64
 5   Manufacturing Site  587392 non-null  str    
 6   division_code       677821 non-null  str    
 7   transaction_date    728463 non-null  str    
 8   Order-Quantity      728463 non-null  int64  
dtypes: float64(1), int64(2), str(6)
memory usage: 50.0 MB


# Question 1: DataFrame formatting

As you load and inspect your transaction dataframe, you'll notice the column names don't follow a consistent convention. Make the necessary changes to ensure all column names follow the same convention as the metadata.

In [4]:
metadata

,feature,description
0,sku_number,Eight character stock keeping unit number
1,inventory_type,Types of inventory including finished good - w...
2,stocking_type,Stockign type policy including make to stock -...
3,lead_time,Total lead time in days
4,unit_price,Unit price US dollar value
5,manufacturing_site,Manufacturing site ID starting with country co...
6,division_code,Four character business division ID
7,transaction_date,Date of Transaction in Year-Month-Day format
8,order_quantity,Order quantity - rounded up for convenience


In [5]:
transactions.columns

Index(['sku_numb3r', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'Manufacturing Site', 'division_code', 'transaction_date',
       'Order-Quantity'],
      dtype='str')

In [6]:
transactions["sku_number"] = transactions["sku_numb3r"]
transactions["manufacturing_site"] = transactions["Manufacturing Site"]
transactions["order_quantity"] = transactions["Order-Quantity"]
transactions = transactions.reindex(columns=["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price",
                                             "manufacturing_site", "division_code", "transaction_date", "order_quantity"])
transactions.columns

Index(['sku_number', 'inventory_type', 'stocking_type', 'lead_time',
       'unit_price', 'manufacturing_site', 'division_code', 'transaction_date',
       'order_quantity'],
      dtype='str')

**Commentary (Q1):** Three names didn't match the metadata (lowercase, underscores). We copied each into a column with the metadata name, then `reindex` kept the nine metadata names in order.

| Original | Problem | New |
|---|---|---|
| `sku_numb3r` | `3` instead of `e` | `sku_number` |
| `Manufacturing Site` | capitals, space | `manufacturing_site` |
| `Order-Quantity` | capitals, hyphen | `order_quantity` |

# Question 2: Data Inspection

Inspect the numeric features of your dataframe with the describe function. Provide a summary of these features, and in case there are any oddities/anomalies in your dataframe, locate and correct them. Please include a note on any oddities you find and a rationale for your suggested correction.

In [7]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,728463.000000,728463.000000,728463.000000
mean,25.782708,800.030488,79.899502
std,10.536974,374.802172,41.920227
min,14.000000,-1012.695895,-126.000000
25%,28.000000,980.472511,64.000000
50%,28.000000,997.457813,93.000000
75%,28.000000,1005.804896,109.000000
max,2800.000000,1026.431584,208.000000


**Commentary (summary):** Problems in `describe()`:

- `lead_time`: max is 2,800 days (normal is 14, 21 or 28).
- `unit_price`: min is -$1,012.70. A price can't be negative.
- `order_quantity`: min is -126. The metadata says quantities are rounded up, so we expect at least 1.

## Oddity 1: `lead_time` values of 1400, 2100 and 2800 days

In [8]:
transactions[transactions["lead_time"] > 28]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
30051,EA4DC0AF,RM,MTS,2800,1012.070796,PL-FA,62D5,2023-02-15,102
47581,E9AF5329,FG,MTS,2800,1014.683386,PL-FA,2875,2023-03-13,85
56045,A963AFB6,FG,MTS,2800,997.839409,JP-C7,62D5,2023-03-25,112
66267,40EBAD87,FG,MTS,2100,194.759445,JP-82,E735,2023-04-10,2
155865,4DB6E386,FG,MTO,2800,996.815534,CA-3F,8CF4,2023-08-23,106
199686,D55CCB7B,FG,MTS,2800,987.397593,JP-82,8CF4,2023-10-28,108
271010,1DF2F967,FG,MTS,2800,1006.255871,US-F1,1ED9,2024-02-12,112
326738,17542C26,WIP,MTS,2100,201.626791,JP-C7,A873,2024-05-06,9
577364,41FB8869,FG,MTO,2800,1011.807794,JP-C7,E735,2025-05-17,91
646672,736308F4,FG,MTS,1400,44.850725,JP-82,5C18,2025-08-29,19


In [9]:
transactions["lead_time"].value_counts()

lead_time
28      583162
14       89014
21       56277
2800         7
2100         2
1400         1
Name: count, dtype: int64

**Commentary:** 10 rows have a `lead_time` of 2,800 (7), 2,100 (2) or 1,400 (1). Each is 100 times a normal value, so we assume two extra zeros were typed. The same SKUs have normal lead times in their other rows (`A963AFB6` is 28 days everywhere else). We fixed them with `replace`.

In [10]:
transactions["lead_time"] = transactions["lead_time"].replace({1400: 14, 2100: 21, 2800: 28})
transactions["lead_time"].value_counts()

lead_time
28    583169
14     89015
21     56279
Name: count, dtype: int64

## Oddity 2: negative `unit_price`

In [11]:
transactions[transactions["unit_price"] < 0]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
70371,F6AC7C21,WIP,MTS,28,-991.288006,US-F1,5D76,2023-04-16,127
76230,4468BB27,FG,MTS,28,-990.568788,PL-FA,62D5,2023-04-25,105
135594,6625477B,WIP,MTS,28,-993.049890,PL-FA,A873,2023-07-23,114
280662,C5D9A49C,WIP,MTO,28,-990.700010,PL-FA,6B4E,2024-02-26,78
393851,1B589A55,FG,MTO,28,-1012.695895,JP-C7,7164,2024-08-15,86
483775,3408C989,FG,MTO,21,-215.230508,US-1F,E735,2024-12-29,-6
508929,706929BB,RM,MTS,21,-188.490312,JP-C7,E735,2025-02-04,55
573230,74684AEA,FG,MTS,28,-1011.897849,JP-C7,762C,2025-05-11,83
671107,3D4ADA3C,FG,MTS,28,-996.250336,PL-FA,C573,2025-10-05,91
713452,4BFB0581,RM,MTS,28,-999.692246,NaN,91A9,2025-12-08,91


In [12]:
transactions[transactions["sku_number"] == "4BFB0581"]["unit_price"].value_counts()

unit_price
 999.692246     1755
 1001.895816    1692
 64.164938        86
-999.692246        1
Name: count, dtype: int64

**Commentary:** 10 rows have a negative `unit_price` (-$188.49 to -$1,012.70). The same SKU has the positive price in its other rows (`4BFB0581`: -$999.69 once, +$999.69 in 1,755 rows), so only the sign is wrong. We fixed them with `abs()`.

In [13]:
transactions["unit_price"] = transactions["unit_price"].abs()
transactions[transactions["unit_price"] < 0].shape

(0, 9)

## Oddity 3: zero and negative `order_quantity`

In [14]:
transactions[transactions["order_quantity"] <= 0]["order_quantity"].value_counts().sort_index()

order_quantity
-126       1
-107       1
-106       1
-102       1
-100       1
        ... 
-4      3225
-3      3501
-2      3679
-1      4042
 0      4222
Name: count, Length: 66, dtype: int64

**Commentary:** 41,779 rows (5.7%) have an `order_quantity` of 0 or less: 37,557 negatives (median -6) and 4,222 zeros. We assume the negatives are returns and keep them (see Assumptions). Q5 includes them.

In [15]:
transactions[transactions["order_quantity"] <= 0].shape

(41779, 9)

## Oddity 4: duplicated transactions

In [16]:
transactions[transactions.duplicated()].sort_values(["transaction_date", "sku_number"]).head(6)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
18933,C1E0F444,FG,MTS,28,1000.141442,CA-3F,E735,2023-01-29,88
21109,F75E5E18,RM,MTS,28,993.353110,NaN,62D5,2023-02-01,83
90721,DB1DC225,RM,MTS,28,993.662818,NaN,2875,2023-05-16,105
92031,09E37F0D,RM,MTO,28,981.151323,US-1F,E122,2023-05-18,93
119634,D266D3E9,FG,MTS,28,1014.496340,PL-FA,AF8D,2023-06-29,85
135439,510ABC31,FG,MTS,28,1010.515648,JP-C7,5D76,2023-07-23,126


In [17]:
transactions[(transactions["sku_number"] == "C1E0F444") & (transactions["transaction_date"] == "2023-01-29")]

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity
18722,C1E0F444,FG,MTO,14,58.432315,JP-C7,E122,2023-01-29,17
18845,C1E0F444,FG,MTS,28,1000.141442,CA-3F,E735,2023-01-29,88
18933,C1E0F444,FG,MTS,28,1000.141442,CA-3F,E735,2023-01-29,88


In [18]:
transactions.duplicated().sum()

np.int64(44)

**Commentary:** 44 rows are identical in all nine columns (`C1E0F444` on 2023-01-29 has two rows of 88 units). There is no transaction ID, and a SKU often has several orders on one day, so these can be real orders. We keep them.

In [19]:
transactions.shape

(728463, 9)

## Numeric features after the corrections

In [20]:
transactions.describe()

,lead_time,unit_price,order_quantity
count,728463.000000,728463.000000,728463.000000
mean,25.748461,800.053522,79.899502
std,4.760940,374.753000,41.920227
min,14.000000,35.786593,-126.000000
25%,28.000000,980.472511,64.000000
50%,28.000000,997.457813,93.000000
75%,28.000000,1005.804896,109.000000
max,28.000000,1026.431584,208.000000


**Commentary (Q2, after):** 728,463 rows. `lead_time` is 14 to 28 days, `unit_price` is $35.79 to $1,026.43 and `order_quantity` is -126 to 208.

# Question 3: NaN Values

Your transactional dataset includes many missing values. Find and address (in your dataframe) the missing values by column. Again, remember to note the NaN values you found and provide a rationale for your suggested action.

In [21]:
transactions.isna().sum()

sku_number              4109
inventory_type         14631
stocking_type           6923
lead_time                  0
unit_price                 0
manufacturing_site    141071
division_code          50642
transaction_date           0
order_quantity             0
dtype: int64

**Commentary:** Five columns have NaNs (`NA` in the raw file).

| Column | NaN | Share of 728,463 rows |
|---|---|---|
| `sku_number` | 4,109 | 0.6% |
| `inventory_type` | 14,631 | 2.0% |
| `stocking_type` | 6,923 | 1.0% |
| `manufacturing_site` | 141,071 | 19.4% |
| `division_code` | 50,642 | 7.0% |

## `sku_number`

**Commentary:** We calculate safety stock per SKU, and no other column identifies it. We dropped the 4,109 rows without a SKU (`notna()`). 724,354 rows left.

In [22]:
transactions = transactions[transactions["sku_number"].notna()]
transactions.shape

(724354, 9)

## `inventory_type`, `stocking_type`, `manufacturing_site` and `division_code`

In [23]:
transactions[transactions["sku_number"] == "00A3840C"]["inventory_type"].value_counts()

inventory_type
FG    3454
RM     189
Name: count, dtype: int64

**Commentary:**

- `inventory_type`, `stocking_type`: filled with `"Unknown"`. The SKU can't fill them because one SKU has several types (`00A3840C` is `FG` in 3,454 rows and `RM` in 189). Q5 leaves these rows out.
- `manufacturing_site` (19.4%): filled with `"Unknown"`. Dropping would lose a fifth of the data.
- `division_code` (7.0%): filled with `"Unknown"`. We don't need it to identify a SKU.

One `fillna` call with a dictionary does all four.

In [24]:
transactions = transactions.fillna({"inventory_type": "Unknown", "stocking_type": "Unknown",
                                    "manufacturing_site": "Unknown", "division_code": "Unknown"})
transactions.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
dtype: int64

In [25]:
transactions.shape

(724354, 9)

**Commentary (Q3, after):** 0 NaNs left. 728,463 minus 4,109 (no SKU) is 724,354 rows.

# Question 4: Extracting The Country Values

To apply broader breakdowns of your transactional dataset, it will be helpful to add a country column. Recall that the metadata indicates the country information is somewhat captured in the site column. Create a new country column by using the `manufacturing_site` information, using the helper function `str.startswith()`.

In [26]:
transactions["manufacturing_site"].value_counts()

manufacturing_site
US-1F      140676
PL-FA      140546
Unknown    140293
JP-C7      140231
US-F1       28419
CA-3F       28357
CA-D0       14067
JP-82       14034
US-A2        7162
CA-D4        7128
PL-D7        7110
JP-6D        7102
CA-FB        7100
CA-6A        7088
CA-24        7084
CA-95        7039
JP-BE        7016
US-4E        6990
PL-51        6912
Name: count, dtype: int64

In [27]:
site = transactions["manufacturing_site"]
transactions["country"] = np.where(site.str.startswith("US-"), "USA",
                          np.where(site.str.startswith("CA-"), "Canada",
                          np.where(site.str.startswith("PL-"), "Poland",
                          np.where(site.str.startswith("JP-"), "Japan", "Unknown"))))
transactions[["manufacturing_site", "country"]].head(10)

,manufacturing_site,country
0,US-1F,USA
1,Unknown,Unknown
2,PL-FA,Poland
3,CA-D0,Canada
4,US-F1,USA
5,PL-FA,Poland
6,US-1F,USA
7,US-1F,USA
8,US-F1,USA
9,PL-FA,Poland


In [28]:
transactions["country"].value_counts()

country
USA        183247
Japan      168383
Poland     154568
Unknown    140293
Canada      77863
Name: count, dtype: int64

**Commentary (Q4):** The metadata says the site ID starts with the country code, and all 18 site codes start with `US-`, `CA-`, `PL-` or `JP-`. We used `str.startswith()` in a nested `np.where`. Anything else is `Unknown`.

| Country | Transactions | Sites |
|---|---|---|
| USA | 183,247 | US-1F, US-F1, US-A2, US-4E |
| Japan | 168,383 | JP-C7, JP-82, JP-6D, JP-BE |
| Poland | 154,568 | PL-FA, PL-D7, PL-51 |
| Canada | 77,863 | CA-3F, CA-D0, CA-D4, CA-FB, CA-6A, CA-24, CA-95 |
| Unknown | 140,293 | sites filled with `"Unknown"` in Q3 |

# Question 5: Helpful Dimensional Information

It's time to focus on the subset of this dataset we need. Since we ultimately want to calculate safety stock, focus on finished goods designated as "make to stock." Create a filtered subset of your transaction dataset and write appropriate code to answer the following questions (for the filtered data).

In [29]:
mts = transactions[(transactions["inventory_type"] == "FG") & (transactions["stocking_type"] == "MTS")]
mts.shape

(267212, 10)

**Commentary (filter):** FG and MTS together give 267,212 of 724,354 rows (36.9%). `"Unknown"` types are excluded.

## 5a. How many unique SKUs are we working with?

In [30]:
mts["sku_number"].value_counts().shape[0]

375

**Commentary (5a):** **375 unique SKUs** (of 624). `value_counts()` lists each SKU once and `.shape[0]` counts them.

## 5b. How many unique Manufacturing Sites are we working with?

In [31]:
mts["manufacturing_site"].value_counts()

manufacturing_site
PL-FA      51903
Unknown    51868
JP-C7      51867
US-1F      51845
CA-3F      10473
US-F1      10442
CA-D0       5103
JP-82       5051
CA-95       2677
CA-FB       2649
CA-24       2645
US-A2       2635
JP-6D       2628
US-4E       2612
PL-D7       2604
JP-BE       2592
CA-6A       2569
CA-D4       2549
PL-51       2500
Name: count, dtype: int64

In [32]:
mts[mts["manufacturing_site"] != "Unknown"]["manufacturing_site"].value_counts().shape[0]

18

**Commentary (5b):** **18 sites**. We removed `"Unknown"` (51,868 rows) first, since it isn't a site.

## 5c. How many Divisions are we working with?

In [33]:
mts["division_code"].value_counts()

division_code
E735       18848
ECB9       18771
E122       18741
8CF4       18670
5D76       18653
62D5       18649
1ED9       18638
2875       18633
Unknown    18608
A873       18516
AF8D       18427
C170        3855
3746        3818
5C18        3747
1F6F        3741
65A4        3686
8E14        3669
49A0        1915
C573        1910
C712        1886
7EA2        1843
A9D5         993
EF3F         987
C26D         978
91A9         977
52DD         972
ED18         970
6B4E         967
F04A         965
D004         963
549A         960
8974         958
3CB5         955
FC8F         955
A1CC         954
228F         948
B0CB         938
762C         938
BAF3         937
5C63         937
FFD5         937
2581         936
7164         935
D8F4         933
611D         929
9A77         923
9AB0         915
45DC         913
49E7         911
EEE7         907
5578         907
A20E         906
41E6         901
0C30         900
D162         883
Name: count, dtype: int64

In [34]:
mts[mts["division_code"] != "Unknown"]["division_code"].value_counts().shape[0]

54

**Commentary (5c):** **54 divisions**. We removed `"Unknown"` (18,608 rows) first.

## 5d. What are the top and bottom 10 transactions by order quantity?

In [35]:
mts.sort_values(["order_quantity", "unit_price", "transaction_date"], ascending=False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
476865,A4DFE12D,FG,MTS,28,1009.801193,US-1F,AF8D,2024-12-18,208,USA
603538,9BD808D9,FG,MTS,28,1012.729872,PL-FA,8CF4,2025-06-26,187,Poland
367041,61A9013F,FG,MTS,28,1003.137627,US-1F,62D5,2024-07-06,186,USA
126203,77074CBD,FG,MTS,28,982.747333,JP-C7,62D5,2023-07-09,186,Japan
236313,A963AFB6,FG,MTS,28,997.839409,US-1F,7EA2,2023-12-22,185,USA
423824,0027AFDA,FG,MTS,28,989.192167,JP-C7,Unknown,2024-09-29,183,Japan
120534,13383A60,FG,MTS,28,996.977138,JP-C7,8CF4,2023-06-30,182,Japan
27638,1CC9F4D5,FG,MTS,28,1014.698985,CA-24,ECB9,2023-02-11,181,Canada
680072,74446B56,FG,MTS,28,998.084102,JP-C7,A873,2025-10-19,181,Japan
303661,96BBC0F5,FG,MTS,28,1010.990593,US-F1,2875,2024-04-01,180,USA


In [36]:
mts.sort_values(["order_quantity", "unit_price", "transaction_date"]).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country
207035,B10F9D77,FG,MTS,28,989.185849,PL-FA,C573,2023-11-08,-106,Poland
248746,44C5014E,FG,MTS,28,986.641044,Unknown,1F6F,2024-01-09,-102,Unknown
642534,F0B94653,FG,MTS,28,995.891524,JP-C7,A873,2025-08-23,-75,Japan
157308,4F772A5B,FG,MTS,21,201.404644,PL-FA,62D5,2023-08-25,-60,Poland
605353,F53926E8,FG,MTS,21,205.986702,Unknown,1ED9,2025-06-28,-52,Unknown
545543,4F772A5B,FG,MTS,21,201.404644,US-1F,1ED9,2025-03-31,-51,USA
300164,8C052224,FG,MTS,21,207.578242,US-1F,C26D,2024-03-27,-49,USA
331630,F1FA52E2,FG,MTS,21,194.501553,JP-82,C170,2024-05-13,-48,Japan
341952,C3F89E8C,FG,MTS,21,205.077972,US-1F,1F6F,2024-05-29,-47,USA
512687,7292F32E,FG,MTS,21,198.325071,PL-FA,8CF4,2025-02-10,-46,Poland


In [37]:
mts[mts["order_quantity"] == 1].shape

(1600, 10)

**Commentary (5d):** Largest order: **208 units** (SKU `A4DFE12D`, `US-1F`, 2024-12-18). Ties are sorted by `unit_price`, then `transaction_date`.

Smallest: **-106 units** (SKU `B10F9D77`). The bottom 10 are all negative (-106 to -46), so we read them as returns. 13,214 rows in this subset are negative and 1,445 are zero. The smallest positive order is 1 unit.

## 5e. What are the top and bottom 10 transactions by total sales value?

Sales is not a feature in this dataset, so we calculate it from the existing features: sales = `unit_price` × `order_quantity`.

In [38]:
mts["sales"] = mts["unit_price"] * mts["order_quantity"]
mts["sales"].describe()

count    267212.000000
mean      78996.076472
std       44235.189745
min     -104853.700012
25%       66086.221936
50%       93177.300909
75%      109601.132855
max      210038.648200
Name: sales, dtype: float64

In [39]:
mts.sort_values(["sales", "transaction_date"], ascending=False).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,sales
476865,A4DFE12D,FG,MTS,28,1009.801193,US-1F,AF8D,2024-12-18,208,USA,210038.648200
603538,9BD808D9,FG,MTS,28,1012.729872,PL-FA,8CF4,2025-06-26,187,Poland,189380.486099
367041,61A9013F,FG,MTS,28,1003.137627,US-1F,62D5,2024-07-06,186,USA,186583.598655
236313,A963AFB6,FG,MTS,28,997.839409,US-1F,7EA2,2023-12-22,185,USA,184600.290718
27638,1CC9F4D5,FG,MTS,28,1014.698985,CA-24,ECB9,2023-02-11,181,Canada,183660.516368
126203,77074CBD,FG,MTS,28,982.747333,JP-C7,62D5,2023-07-09,186,Japan,182791.004027
303661,96BBC0F5,FG,MTS,28,1010.990593,US-F1,2875,2024-04-01,180,USA,181978.306814
120534,13383A60,FG,MTS,28,996.977138,JP-C7,8CF4,2023-06-30,182,Japan,181449.839088
448644,1DF2F967,FG,MTS,28,1006.255871,PL-FA,2875,2024-11-05,180,Poland,181126.056746
423824,0027AFDA,FG,MTS,28,989.192167,JP-C7,Unknown,2024-09-29,183,Japan,181022.166518


In [40]:
mts.sort_values(["sales", "transaction_date"]).head(10)

,sku_number,inventory_type,stocking_type,lead_time,unit_price,manufacturing_site,division_code,transaction_date,order_quantity,country,sales
207035,B10F9D77,FG,MTS,28,989.185849,PL-FA,C573,2023-11-08,-106,Poland,-104853.700012
248746,44C5014E,FG,MTS,28,986.641044,Unknown,1F6F,2024-01-09,-102,Unknown,-100637.386536
642534,F0B94653,FG,MTS,28,995.891524,JP-C7,A873,2025-08-23,-75,Japan,-74691.864325
157308,4F772A5B,FG,MTS,21,201.404644,PL-FA,62D5,2023-08-25,-60,Poland,-12084.278653
605353,F53926E8,FG,MTS,21,205.986702,Unknown,1ED9,2025-06-28,-52,Unknown,-10711.308500
545543,4F772A5B,FG,MTS,21,201.404644,US-1F,1ED9,2025-03-31,-51,USA,-10271.636855
300164,8C052224,FG,MTS,21,207.578242,US-1F,C26D,2024-03-27,-49,USA,-10171.333849
233858,417E5AEE,FG,MTS,21,219.200471,PL-FA,ECB9,2023-12-18,-44,Poland,-9644.820704
341952,C3F89E8C,FG,MTS,21,205.077972,US-1F,1F6F,2024-05-29,-47,USA,-9638.664693
34474,D85CEB96,FG,MTS,21,207.996884,PL-FA,E735,2023-02-21,-45,Poland,-9359.859766


**Commentary (5e):** Largest sale: **$210,038.65**, the same order as the largest quantity. Smallest: **-$104,853.70** (SKU `B10F9D77`, -106 units). The bottom 10 are all negative (down to -$9,359.86), so we read them as refunds, not small sales.

# Assumptions

- Rows with an `order_quantity` of 0 or less (41,779) might be returns. We keep them, and the Q5 bottom 10 includes them.
- The 44 identical rows might be real orders. We keep them.
- 10 lead times of 1,400, 2,100 and 2,800 are typing errors (100 times too big). We corrected them.
- 10 negative prices have the wrong sign. We made them positive.
- Rows with no SKU can't be matched to a product. We dropped them. Other blanks are filled with `"Unknown"`.
- We have no more context yet. If later assignments need it, we will change these.

# OM 620 Assignment 2: Safety Stock Calculation

OM 620 Tools and Technologies for Analytics, California State University San Marcos, Fall 2026.

Group members: Suleiman Sahawneh, Jorge, Daniel, Diego.

In Assignment 1 we cleaned the transaction data and found the finished goods that are made to stock (MTS). In this assignment we summarize demand for each of those SKUs and calculate how much safety stock each one needs at three service levels.

# Setup

We use `pandas` to load, filter and summarize the data, and `numpy` for `np.where` in the country step we reuse from Assignment 1. From SciPy we load `norm` for the z scores in Question 2 and `mquantiles` for the empirical quantiles in the side quest.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import norm
from scipy.stats.mstats import mquantiles

## Loading the data

Our group's dataset (`transaction_data.csv`) is in the `data` subfolder, the same file we used in Assignment 1.

In [2]:
transactions = pd.read_csv("../data/transaction_data.csv")
transactions.shape

(728463, 9)

# Rebuilding the Assignment 1 data

We run our Assignment 1 cleaning code again, with one change from the professor's feedback on Assignment 1. The steps are:

1. Copy the three badly named columns into the metadata names and keep the nine columns in order.
2. Correct the lead times typed as 1400, 2100 and 2800 days to 14, 21 and 28.
3. Make the ten negative unit prices positive.
4. Make the negative order quantities positive. In Assignment 1 we kept them as returns. The professor's feedback said they are most likely sign errors, like the unit prices, so the absolute value is the correct fix.
5. Drop the 4,109 rows with no SKU number and fill the other blanks with `"Unknown"`.
6. Build the `country` column from the manufacturing site.

As in Assignment 1, rows with a quantity of 0 stay in, and identical rows stay in as possible repeat orders.

In [3]:
transactions["sku_number"] = transactions["sku_numb3r"]
transactions["manufacturing_site"] = transactions["Manufacturing Site"]
transactions["order_quantity"] = transactions["Order-Quantity"]
transactions = transactions.reindex(columns=["sku_number", "inventory_type", "stocking_type", "lead_time", "unit_price",
                                             "manufacturing_site", "division_code", "transaction_date", "order_quantity"])

transactions["lead_time"] = transactions["lead_time"].replace({1400: 14, 2100: 21, 2800: 28})
transactions["unit_price"] = transactions["unit_price"].abs()
transactions["order_quantity"] = transactions["order_quantity"].abs()

transactions = transactions[transactions["sku_number"].notna()]
transactions = transactions.fillna({"inventory_type": "Unknown", "stocking_type": "Unknown",
                                    "manufacturing_site": "Unknown", "division_code": "Unknown"})

site = transactions["manufacturing_site"]
transactions["country"] = np.where(site.str.startswith("US-"), "USA",
                          np.where(site.str.startswith("CA-"), "Canada",
                          np.where(site.str.startswith("PL-"), "Poland",
                          np.where(site.str.startswith("JP-"), "Japan", "Unknown"))))
transactions.shape

(724354, 10)

In [4]:
transactions.isna().sum()

sku_number            0
inventory_type        0
stocking_type         0
lead_time             0
unit_price            0
manufacturing_site    0
division_code         0
transaction_date      0
order_quantity        0
country               0
dtype: int64

**Commentary:** The cleaned table has 724,354 rows and no missing values, the same result as Assignment 1.

## Filtering for finished goods made to stock

Safety stock applies to items we keep on the shelf before anyone orders them, so we keep only rows that are finished goods (`FG`) and made to stock (`MTS`).

In [5]:
mts = transactions[(transactions["inventory_type"] == "FG") & (transactions["stocking_type"] == "MTS")]
mts.shape

(267212, 10)

In [6]:
mts["sku_number"].value_counts().shape[0]

375

In [7]:
mts[mts["order_quantity"] < 0].shape[0], mts[mts["order_quantity"] == 0].shape[0]

(0, 1445)

**Commentary:** The filter keeps 267,212 transactions for 375 SKUs, which matches Assignment 1. After the absolute value fix there are no negative quantities left. The 1,445 zero quantities stay in.

# Question 1: Required Transformation

Create a transformed dataset to filter for appropriate SKUs that are also aggregated with the following statistics: minimum, maximum, average, median, variance, and standard deviation of quantity, along with average lead time.

In [8]:
sku_stats = (
    mts
    .groupby("sku_number")
    .agg(
        min_quantity=("order_quantity", "min"),
        max_quantity=("order_quantity", "max"),
        mean_quantity=("order_quantity", "mean"),
        median_quantity=("order_quantity", "median"),
        var_quantity=("order_quantity", "var"),
        std_quantity=("order_quantity", "std"),
        mean_lead_time=("lead_time", "mean"),
    )
)
sku_stats.round(2)

,min_quantity,max_quantity,mean_quantity,median_quantity,var_quantity,std_quantity,mean_lead_time
sku_number,,,,,,,
0027AFDA,39,183,100.55,101.0,398.99,19.97,28.00
01637C60,0,35,8.04,7.0,47.17,6.87,14.00
02F3C196,0,30,8.39,7.0,37.39,6.11,14.00
04711B22,0,25,8.24,7.0,33.73,5.81,14.00
05B8D97A,23,160,100.42,100.0,374.61,19.35,28.00
...,...,...,...,...,...,...,...
FD350FE2,0,169,87.33,96.0,1340.90,36.62,26.04
FD3CB2F5,13,161,99.48,100.0,418.05,20.45,28.00
FDC306C8,1,29,8.94,8.0,41.08,6.41,14.00


In [9]:
sku_stats.shape

(375, 7)

In [10]:
sku_stats.describe().round(2)

,min_quantity,max_quantity,mean_quantity,median_quantity,var_quantity,std_quantity,mean_lead_time
count,375.00,375.00,375.00,375.00,375.00,375.00,375.00
mean,4.08,78.18,36.51,36.22,271.07,13.64,19.47
std,10.78,61.54,38.95,40.89,352.45,9.24,6.17
min,0.00,22.00,7.54,6.00,28.64,5.35,14.00
25%,0.00,30.00,8.92,7.50,44.15,6.64,14.00
50%,0.00,37.00,10.05,9.00,55.69,7.46,14.00
75%,0.00,161.00,89.61,97.00,396.54,19.91,27.31
max,43.00,208.00,100.68,101.00,1776.10,42.14,28.00


In [11]:
mts["sku_number"].value_counts().min()

np.int64(65)

In [12]:
(sku_stats["min_quantity"] == 0).sum()

np.int64(309)

In [13]:
lead_check = mts.groupby("sku_number").agg(
    lowest_lead_time=("lead_time", "min"),
    highest_lead_time=("lead_time", "max"),
)
(lead_check["lowest_lead_time"] != lead_check["highest_lead_time"]).sum()

np.int64(87)

In [14]:
sku_stats.groupby(sku_stats["std_quantity"] > 15).agg(
    skus=("std_quantity", "size"),
    lowest_std=("std_quantity", "min"),
    highest_std=("std_quantity", "max"),
    average_quantity=("mean_quantity", "mean"),
    average_lead_time=("mean_lead_time", "mean"),
).round(2)

,skus,lowest_std,highest_std,average_quantity,average_lead_time
std_quantity,,,,,
False,225,5.35,14.85,9.45,15.29
True,150,15.09,42.14,77.08,25.75


**Commentary (Q1):** We group the 267,212 MTS transactions by `sku_number` and use named aggregation, so each of the seven statistics gets its own column. The result has 375 rows, one per SKU. The SKU with the fewest transactions still has 65, so every SKU has a variance and a standard deviation.

* 309 of the 375 SKUs have a minimum of 0. These are the zero quantities we kept from Assignment 1.
* We use a standard deviation of 15 units as the cut because it separates small orders from large ones. The 225 SKUs with a standard deviation from 5.35 to 14.85 units order about 9 units at a time and have an average lead time of about 15 days. The 150 SKUs with a standard deviation from 15.09 to 42.14 units order about 77 units at a time and wait about 26 days.
* 87 SKUs were shipped with more than one lead time (14, 21 or 28 days), so some average lead times are not whole numbers.

# Question 2: Safety Stock Calculation

Use the transformed data from Question 1 to calculate three safety stock values for each SKU for the service level coefficients of 75%, 90%, and 95%.

We use the formula from the assignment, $SS = z_{\alpha} \times \sigma_D \times \sqrt{L}$, with each SKU's average lead time in days as $L$. The assignment says to use the Question 1 table, so $\sigma_D$ is each SKU's standard deviation of quantity per order, not per day. Multiplying by $\sqrt{L}$ with $L$ in days therefore treats each order as one day of demand. We take the square root with the power operator `** 0.5`.

In [15]:
z_75 = norm.ppf(0.75)
z_90 = norm.ppf(0.90)
z_95 = norm.ppf(0.95)
z_75, z_90, z_95

(np.float64(0.6744897501960817),
 np.float64(1.2815515655446004),
 np.float64(1.6448536269514722))

In [16]:
safety_stock = sku_stats.assign(
    ss_75=z_75 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
    ss_90=z_90 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
    ss_95=z_95 * sku_stats["std_quantity"] * sku_stats["mean_lead_time"] ** 0.5,
)
safety_stock[["std_quantity", "mean_lead_time", "ss_75", "ss_90", "ss_95"]].round(2)

,std_quantity,mean_lead_time,ss_75,ss_90,ss_95
sku_number,,,,,
0027AFDA,19.97,28.00,71.29,135.46,173.86
01637C60,6.87,14.00,17.33,32.93,42.27
02F3C196,6.11,14.00,15.43,29.32,37.63
04711B22,5.81,14.00,14.66,27.85,35.74
05B8D97A,19.35,28.00,69.08,131.25,168.46
...,...,...,...,...,...
FD350FE2,36.62,26.04,126.04,239.48,307.37
FD3CB2F5,20.45,28.00,72.97,138.65,177.96
FDC306C8,6.41,14.00,16.18,30.73,39.45


In [17]:
safety_stock[["ss_75", "ss_90", "ss_95"]].describe().round(2)

,ss_75,ss_90,ss_95
count,375.00,375.00,375.00
mean,43.64,82.92,106.42
std,34.96,66.42,85.26
min,13.51,25.66,32.94
25%,16.77,31.86,40.89
50%,19.24,36.55,46.91
75%,71.07,135.04,173.32
max,144.81,275.14,353.14


**Commentary (Q2):** `norm.ppf` gives z scores of 0.674 for 75%, 1.282 for 90% and 1.645 for 95%. Only z changes between the three columns, so each SKU's 95% safety stock is 2.44 times its 75% value (1.645 divided by 0.674), and the SKUs keep the same order at every level.

The average safety stock rises from 43.64 units at 75% to 82.92 at 90% and 106.42 at 95%. Going from 90% to 95% adds 23.50 units per SKU on average, which is the cost of cutting the stockout risk from 10% to 5%. In the summary, the 95% column jumps from 46.91 units at the median to 173.32 at the 75th percentile.

# Question 3: Safety Stock Distribution

Using the 95% service level coefficient, which SKU has the largest safety stock, which SKU has the smallest, and what is the average safety stock value across all SKUs?

## Largest safety stock

In [18]:
safety_stock.sort_values("ss_95", ascending=False).head(1)[["mean_quantity", "std_quantity", "mean_lead_time", "ss_95"]]

,mean_quantity,std_quantity,mean_lead_time,ss_95
sku_number,,,,
084B19C4,75.108714,42.143834,25.952282,353.14176


## Smallest safety stock

In [19]:
safety_stock.sort_values("ss_95").head(1)[["mean_quantity", "std_quantity", "mean_lead_time", "ss_95"]]

,mean_quantity,std_quantity,mean_lead_time,ss_95
sku_number,,,,
FE8A9BBE,7.542169,5.351775,14.0,32.937384


## Average safety stock across all SKUs

In [20]:
safety_stock["ss_95"].mean()

np.float64(106.42146831675336)

In [21]:
(safety_stock["ss_95"] < safety_stock["ss_95"].mean()).sum()

np.int64(222)

In [22]:
(
    safety_stock
    .assign(gap_to_average=(safety_stock["ss_95"] - safety_stock["ss_95"].mean()).abs())
    .sort_values("gap_to_average")
    .head(1)[["std_quantity", "mean_lead_time", "ss_95", "gap_to_average"]]
)

,std_quantity,mean_lead_time,ss_95,gap_to_average
sku_number,,,,
92EFFC31,14.842193,18.502846,105.013374,1.408094


**Commentary (Q3):**

* The largest safety stock belongs to SKU `084B19C4`, at 353.14 units. It has the highest standard deviation of all 375 SKUs (42.14 units) and an average lead time of 25.95 days.
* The smallest belongs to SKU `FE8A9BBE`, at 32.94 units. It has the lowest standard deviation (5.35 units) and the minimum lead time of 14 days.
* The average across all SKUs is 106.42 units. Because of the two groups from Question 1, 222 SKUs are below it and the median is only 46.91. The closest SKU to the average is `92EFFC31`, at 105.01 units, 1.41 below it.

The largest value is about 11 times the smallest. If we gave every SKU the average of 106.42 units, the 153 SKUs above the average would get less than their 95% safety stock and the 222 SKUs below it would hold more stock than they need.

## Checking the zero quantities

Our `mts` rows still include the 1,445 zero quantities we kept in Assignment 1. A zero order may not be real demand, so we recalculate the 95% safety stock with only the orders above zero to see how much our answers depend on keeping them.

In [23]:
orders_only = mts[mts["order_quantity"] > 0].groupby("sku_number").agg(
    std_quantity=("order_quantity", "std"),
    mean_lead_time=("lead_time", "mean"),
)
orders_only = orders_only.assign(ss_95=z_95 * orders_only["std_quantity"] * orders_only["mean_lead_time"] ** 0.5)
orders_only.shape

(375, 3)

In [24]:
pd.DataFrame(
    {
        "with_zeros": [safety_stock["ss_95"].max(), safety_stock["ss_95"].min(), safety_stock["ss_95"].mean()],
        "orders_only": [orders_only["ss_95"].max(), orders_only["ss_95"].min(), orders_only["ss_95"].mean()],
    },
    index=["largest", "smallest", "average"],
).round(2)

,with_zeros,orders_only
largest,353.14,350.63
smallest,32.94,31.72
average,106.42,105.58


In [25]:
orders_only.sort_values("ss_95", ascending=False).head(1)

,std_quantity,mean_lead_time,ss_95
sku_number,,,
084B19C4,41.813248,25.990389,350.628773


In [26]:
orders_only.sort_values("ss_95").head(1)

,std_quantity,mean_lead_time,ss_95
sku_number,,,
FE8A9BBE,5.154684,14.0,31.72439


**Commentary (zero quantities check):** All 375 SKUs still have orders above zero. Without the zeros, the largest safety stock is still `084B19C4` (350.63 units instead of 353.14) and the smallest is still `FE8A9BBE` (31.72 instead of 32.94). The average falls from 106.42 to 105.58 units, less than one unit per SKU. Keeping the zeros does not change which SKUs are the largest and the smallest, so we keep them, as in Assignment 1.

# Side Quest: Non parametric Safety Stock

The formula above assumes demand for each SKU is normally distributed. Question 1 showed that 87 SKUs were shipped with more than one lead time. If a SKU's order size depends on its lead time, its orders come from more than one pattern and will not follow a single bell curve. Here we drop the normal assumption and read the quantiles straight from each SKU's own transactions with `mquantiles`.

Under the normal model, $z_{\alpha} \times \sigma_D$ is the distance from the average demand to the $\alpha$ quantile of demand, which is the gap between the two lines in the assignment's figure. We measure the same gap without the normal model: we take the empirical $\alpha$ quantile of each SKU's order quantities, subtract the SKU's average quantity, and scale by $\sqrt{L}$ as before.

$$SS_{\text{empirical}} = (Q_{\alpha} - \bar{D}) \times \sqrt{L}$$

`mquantiles` uses its default plotting positions (`alphap=0.4`, `betap=0.4`), so a quantile can fall between two observed quantities.

In [27]:
quantiles = mts.groupby("sku_number").agg(
    q_75=("order_quantity", lambda x: mquantiles(x, prob=[0.75])[0]),
    q_90=("order_quantity", lambda x: mquantiles(x, prob=[0.90])[0]),
    q_95=("order_quantity", lambda x: mquantiles(x, prob=[0.95])[0]),
)
empirical = sku_stats.assign(q_75=quantiles["q_75"], q_90=quantiles["q_90"], q_95=quantiles["q_95"])
empirical = empirical.assign(
    ss_emp_75=(empirical["q_75"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
    ss_emp_90=(empirical["q_90"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
    ss_emp_95=(empirical["q_95"] - empirical["mean_quantity"]) * empirical["mean_lead_time"] ** 0.5,
)
empirical[["mean_quantity", "q_75", "q_90", "q_95", "ss_emp_75", "ss_emp_90", "ss_emp_95"]].round(2)

,mean_quantity,q_75,q_90,q_95,ss_emp_75,ss_emp_90,ss_emp_95
sku_number,,,,,,,
0027AFDA,100.55,114.0,126.00,133.00,71.15,134.64,171.69
01637C60,8.04,10.0,16.18,22.17,7.35,30.47,52.89
02F3C196,8.39,13.0,16.28,19.94,17.26,29.53,43.23
04711B22,8.24,13.0,16.08,18.68,17.81,29.33,39.06
05B8D97A,100.42,113.0,126.00,133.00,66.56,135.35,172.39
...,...,...,...,...,...,...,...
FD350FE2,87.33,111.0,124.00,131.00,120.77,187.11,222.83
FD3CB2F5,99.48,114.0,126.00,133.00,76.85,140.35,177.39
FDC306C8,8.94,13.0,17.56,21.00,15.21,32.27,45.14


## Largest, smallest and average empirical safety stock (95%)

In [28]:
empirical.sort_values("ss_emp_95", ascending=False).head(1)[["mean_quantity", "median_quantity", "q_95", "ss_emp_95"]]

,mean_quantity,median_quantity,q_95,ss_emp_95
sku_number,,,,
084B19C4,75.108714,89.0,130.0,279.634779


In [29]:
mts[mts["sku_number"] == "084B19C4"].groupby("lead_time").agg(
    orders=("order_quantity", "size"),
    average_quantity=("order_quantity", "mean"),
).round(2)

,orders,average_quantity
lead_time,,
14,183,9.72
21,339,23.12
28,1888,90.78


In [30]:
empirical.sort_values("ss_emp_95").head(1)[["mean_quantity", "median_quantity", "q_95", "ss_emp_95"]]

,mean_quantity,median_quantity,q_95,ss_emp_95
sku_number,,,,
A6AB6A6D,7.978495,7.0,16.94,33.530883


In [31]:
empirical["ss_emp_95"].mean()

np.float64(103.0585735166693)

In [32]:
(empirical["q_95"] * empirical["mean_lead_time"] ** 0.5).mean()

np.float64(285.5372004767801)

## Comparing the two methods

In [33]:
pd.DataFrame(
    {
        "normal_average": [safety_stock["ss_75"].mean(), safety_stock["ss_90"].mean(), safety_stock["ss_95"].mean()],
        "empirical_average": [empirical["ss_emp_75"].mean(), empirical["ss_emp_90"].mean(), empirical["ss_emp_95"].mean()],
    },
    index=["75%", "90%", "95%"],
).round(2)

,normal_average,empirical_average
75%,43.64,41.32
90%,82.92,79.78
95%,106.42,103.06


In [34]:
comparison = safety_stock[["std_quantity", "ss_95"]].assign(
    ss_emp_95=empirical["ss_emp_95"],
    difference=safety_stock["ss_95"] - empirical["ss_emp_95"],
)
comparison.loc[["084B19C4", "FE8A9BBE", "A6AB6A6D"]].round(2)

,std_quantity,ss_95,ss_emp_95,difference
sku_number,,,,
084B19C4,42.14,353.14,279.63,73.51
FE8A9BBE,5.35,32.94,37.03,-4.10
A6AB6A6D,5.81,35.74,33.53,2.21


In [35]:
comparison.sort_values("difference", ascending=False).head(3).round(2)

,std_quantity,ss_95,ss_emp_95,difference
sku_number,,,,
7217A16C,37.06,310.94,217.02,93.92
DCFACBBC,38.72,330.19,237.27,92.92
FD350FE2,36.62,307.37,222.83,84.54


In [36]:
(comparison["ss_emp_95"] > comparison["ss_95"]).sum(), (comparison["ss_emp_95"] < comparison["ss_95"]).sum()

(np.int64(276), np.int64(99))

In [37]:
comparison.groupby(comparison["std_quantity"] > 15).agg(
    skus=("ss_95", "size"),
    normal_average=("ss_95", "mean"),
    empirical_average=("ss_emp_95", "mean"),
).round(2)

,skus,normal_average,empirical_average
std_quantity,,,
False,225,45.54,52.60
True,150,197.74,178.75


In [38]:
high_group = comparison[comparison["std_quantity"] > 15].assign(
    mixed_lead_time=lead_check["lowest_lead_time"] != lead_check["highest_lead_time"],
)
high_group.groupby("mixed_lead_time").agg(
    skus=("ss_95", "size"),
    normal_average=("ss_95", "mean"),
    empirical_average=("ss_emp_95", "mean"),
    average_difference=("difference", "mean"),
).round(2)

,skus,normal_average,empirical_average,average_difference
mixed_lead_time,,,,
False,79,163.52,165.03,-1.51
True,71,235.81,194.01,41.79


**Commentary (side quest):**

* `084B19C4` is still the largest, but the empirical method gives 279.63 units instead of 353.14. Its order size depends on the lead time: its 183 orders shipped in 14 days average 9.72 units, its 339 orders shipped in 21 days average 23.12 units, and its 1,888 orders shipped in 28 days average 90.78 units. Mixing these puts its median (89 units) above its average (75.11 units) and inflates its standard deviation, so the normal formula overstates how far its orders climb above the average.
* The smallest is now `A6AB6A6D` at 33.53 units. `FE8A9BBE`, the smallest under the normal formula, moves up to 37.03 units from 32.94.
* The average is 103.06 units per SKU, about 3% below the 106.42 from the normal formula. It is also lower at 75% (41.32 against 43.64) and at 90% (79.78 against 82.92). Using the 95% quantile itself, without subtracting the average, would give 285.54 units. That counts each SKU's average order as safety stock, while the formula covers only the distance above the average (the gap between the two lines in the assignment's figure).

The small gap in the average hides two opposite effects. For the 225 SKUs in the low group, the empirical method asks for more (52.60 against 45.54 units on average). Their own order history reaches further above the average than the normal formula assumes, so the formula gives them less stock than that history calls for. For the 150 SKUs in the high group, it asks for about 19 fewer units each (178.75 against 197.74). That gap comes from the 71 high group SKUs with more than one lead time, where the normal value is 41.79 units higher on average. For the 79 high group SKUs with a single lead time, the empirical value is 1.51 units higher. The three largest gaps, led by `7217A16C` at 93.92 units, are all large SKUs with standard deviations between 36.62 and 38.72 units. Across all SKUs the empirical value is higher for 276 and lower for 99.

For the 71 SKUs with mixed lead times we would trust the empirical values more, because their gap suggests that mixing lead times drives their spread, as it does for `084B19C4`. A further step would be to calculate their safety stock separately for each lead time. The method also needs enough history: the SKU with the fewest transactions has 65, which leaves only about three orders above its 95% quantile.

# Reference

* OM/OSBA 620 course lectures and lecture notebooks (CSUSM, Fall 2026), in particular Data Transformation (grouping and summarizing with `groupby` and `agg`) and Exploratory Data Analysis (density and service level discussion).
* OM 620 Assignment 2 description, including the safety stock formula, `ppf` from `scipy.stats`, and `mquantiles` from `scipy.stats.mstats`.
* Our own Assignment 1 notebook, for the cleaning code reused above.
* The professor's feedback on our Assignment 1, which said to use absolute values for the negative order quantities.
* SciPy documentation for `scipy.stats.norm` and `scipy.stats.mstats.mquantiles` (default `alphap` and `betap` values): https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mstats.mquantiles.html